In [8]:
# Cell 0 — Project Imports

import torch

In [9]:
# Cell 1 — Dataset Geometry Fingerprint

def summarize_dataset_geometry(
    case_shapes_zyx: torch.Tensor,       # [N, 3], integer
    case_spacings_zyx_mm: torch.Tensor,  # [N, 3], floating-point
) -> dict[str, torch.Tensor]:
    """Dataset geometry의 핵심 통계 계산."""

    # 각 행이 한 환자의 (D, H, W)를 나타내는지 확인
    if (
        case_shapes_zyx.ndim != 2
        or case_shapes_zyx.shape[1] != 3
    ):
        raise ValueError(
            "case_shapes_zyx의 Shape는 [N, 3]이어야 합니다."
        )

    # Shape와 spacing이 동일한 환자 수를 가지는지 확인
    if case_shapes_zyx.shape != case_spacings_zyx_mm.shape:
        raise ValueError(
            "Shape와 spacing Tensor의 Shape가 일치해야 합니다."
        )

    # Voxel 개수를 나타내는 integer Shape인지 확인
    if torch.is_floating_point(case_shapes_zyx):
        raise TypeError(
            "case_shapes_zyx는 integer Tensor여야 합니다."
        )

    # Millimeter 단위 계산이 가능한 floating-point spacing인지 확인
    if not torch.is_floating_point(case_spacings_zyx_mm):
        raise TypeError(
            "case_spacings_zyx_mm는 floating-point Tensor여야 합니다."
        )

    # 0 이하의 Shape 또는 spacing 차단
    if not bool(
        torch.all(case_shapes_zyx > 0).item()
    ):
        raise ValueError(
            "모든 voxel Shape 값은 양수여야 합니다."
        )

    if not bool(
        torch.all(case_spacings_zyx_mm > 0).item()
    ):
        raise ValueError(
            "모든 spacing 값은 양수여야 합니다."
        )

    # Voxel 개수를 floating-point로 변환
    case_shapes_float = case_shapes_zyx.to(
        torch.float32
    )  # [N, 3]

    # 환자별 실제 촬영 범위 계산
    case_physical_extents_zyx_mm = (
        case_shapes_float
        * case_spacings_zyx_mm
    )  # [N, 3]

    # 각 환자의 가장 큰 spacing과 가장 작은 spacing의 비율 계산
    case_anisotropy_ratios = (
        case_spacings_zyx_mm.max(dim=1).values
        / case_spacings_zyx_mm.min(dim=1).values
    )  # [N]

    # Dataset의 대표 geometry를 위한 axis별 중앙값 계산
    median_shape_zyx = torch.quantile(
        case_shapes_float,
        q=0.5,
        dim=0,
    )  # [3]

    median_spacing_zyx_mm = torch.quantile(
        case_spacings_zyx_mm,
        q=0.5,
        dim=0,
    )  # [3]

    median_physical_extent_zyx_mm = torch.quantile(
        case_physical_extents_zyx_mm,
        q=0.5,
        dim=0,
    )  # [3]

    return {
        "physical_extents_zyx_mm": (
            case_physical_extents_zyx_mm
        ),
        "anisotropy_ratios": case_anisotropy_ratios,
        "median_shape_zyx": median_shape_zyx,
        "median_spacing_zyx_mm": median_spacing_zyx_mm,
        "median_physical_extent_zyx_mm": (
            median_physical_extent_zyx_mm
        ),
    }
    
# 네 환자의 가상 CT voxel Shape 구성
synthetic_case_shapes_zyx = torch.tensor(
    [
        [96, 256, 256],
        [120, 240, 256],
        [88, 272, 240],
        [104, 256, 272],
    ],
    dtype=torch.int64,
)  # [N=4, 3]


# 같은 환자들의 (z, y, x) voxel spacing 구성
synthetic_case_spacings_zyx_mm = torch.tensor(
    [
        [3.0, 1.0, 1.0],
        [2.5, 1.2, 1.0],
        [4.0, 0.9, 0.9],
        [3.5, 1.0, 1.1],
    ],
    dtype=torch.float32,
)  # [N=4, 3]


geometry_fingerprint = summarize_dataset_geometry(
    case_shapes_zyx=synthetic_case_shapes_zyx,
    case_spacings_zyx_mm=synthetic_case_spacings_zyx_mm,
)


# 환자별 voxel geometry와 physical geometry 비교
for case_index in range(
    synthetic_case_shapes_zyx.shape[0]
):
    shape_zyx = synthetic_case_shapes_zyx[
        case_index
    ]

    spacing_zyx_mm = synthetic_case_spacings_zyx_mm[
        case_index
    ]

    physical_extent_zyx_mm = geometry_fingerprint[
        "physical_extents_zyx_mm"
    ][case_index]

    anisotropy_ratio = geometry_fingerprint[
        "anisotropy_ratios"
    ][case_index]

    print(
        f"Case {case_index} | "
        f"shape={shape_zyx.tolist()} | "
        f"spacing={spacing_zyx_mm.tolist()} mm | "
        f"extent={physical_extent_zyx_mm.tolist()} mm | "
        f"anisotropy={anisotropy_ratio.item():.2f}"
    )


# Dataset 전체의 대표 geometry 통계 출력
print()
print(
    "Median shape:",
    geometry_fingerprint[
        "median_shape_zyx"
    ].tolist(),
)

print(
    "Median spacing:",
    geometry_fingerprint[
        "median_spacing_zyx_mm"
    ].tolist(),
    "mm",
)

print(
    "Median physical extent:",
    geometry_fingerprint[
        "median_physical_extent_zyx_mm"
    ].tolist(),
    "mm",
)

Case 0 | shape=[96, 256, 256] | spacing=[3.0, 1.0, 1.0] mm | extent=[288.0, 256.0, 256.0] mm | anisotropy=3.00
Case 1 | shape=[120, 240, 256] | spacing=[2.5, 1.2000000476837158, 1.0] mm | extent=[300.0, 288.0, 256.0] mm | anisotropy=2.50
Case 2 | shape=[88, 272, 240] | spacing=[4.0, 0.8999999761581421, 0.8999999761581421] mm | extent=[352.0, 244.79998779296875, 216.0] mm | anisotropy=4.44
Case 3 | shape=[104, 256, 272] | spacing=[3.5, 1.0, 1.100000023841858] mm | extent=[364.0, 256.0, 299.20001220703125] mm | anisotropy=3.50

Median shape: [100.0, 256.0, 256.0]
Median spacing: [3.25, 1.0, 1.0] mm
Median physical extent: [326.0, 256.0, 256.0] mm


In [10]:
# Cell 2 — Fingerprint와 Experiment Plans 역할 분리

def compute_planned_patch_context(
    target_spacing_zyx_mm: torch.Tensor,  # [3], floating-point
    patch_size_zyx: torch.Tensor,         # [3], integer
) -> tuple[
    torch.Tensor,  # Physical field of view [3]
    int,           # Patch voxel count
    torch.Tensor,  # Physical patch volume []
]:
    """Planned patch의 voxel·physical context 계산."""

    # (z, y, x) 세 axis를 가진 1D Tensor인지 확인
    if target_spacing_zyx_mm.shape != (3,):
        raise ValueError(
            "target_spacing_zyx_mm의 Shape는 [3]이어야 합니다."
        )

    if patch_size_zyx.shape != (3,):
        raise ValueError(
            "patch_size_zyx의 Shape는 [3]이어야 합니다."
        )

    # Physical-space 계산이 가능한 floating-point spacing인지 확인
    if not torch.is_floating_point(
        target_spacing_zyx_mm
    ):
        raise TypeError(
            "target spacing은 floating-point Tensor여야 합니다."
        )

    # Voxel 개수를 나타내는 integer patch size인지 확인
    if torch.is_floating_point(
        patch_size_zyx
    ):
        raise TypeError(
            "patch size는 integer Tensor여야 합니다."
        )

    # 잘못된 0 이하의 spacing과 patch size 차단
    if not bool(
        torch.all(target_spacing_zyx_mm > 0).item()
    ):
        raise ValueError(
            "모든 target spacing은 양수여야 합니다."
        )

    if not bool(
        torch.all(patch_size_zyx > 0).item()
    ):
        raise ValueError(
            "모든 patch size는 양수여야 합니다."
        )

    # Planned patch가 실제 공간에서 포함하는 축별 길이 계산
    physical_field_of_view_zyx_mm = (
        patch_size_zyx.to(torch.float32)
        * target_spacing_zyx_mm
    )  # [3]

    # 한 patch에 포함되는 전체 voxel 수 계산
    patch_voxel_count = int(
        torch.prod(
            patch_size_zyx
        ).item()
    )

    # 한 patch가 포함하는 실제 부피 계산
    physical_patch_volume_mm3 = torch.prod(
        physical_field_of_view_zyx_mm
    )  # []

    return (
        physical_field_of_view_zyx_mm,
        patch_voxel_count,
        physical_patch_volume_mm3,
    )


# Cell 1에서 관찰한 Dataset geometry fingerprint 구성
dataset_fingerprint: dict[
    str,
    torch.Tensor,
] = {
    "median_shape_zyx": geometry_fingerprint[
        "median_shape_zyx"
    ],
    "median_spacing_zyx_mm": geometry_fingerprint[
        "median_spacing_zyx_mm"
    ],
    "median_physical_extent_zyx_mm": (
        geometry_fingerprint[
            "median_physical_extent_zyx_mm"
        ]
    ),
}


# Planner가 결정했다고 가정한 교육용 experiment plans 구성
experiment_plans: dict[
    str,
    torch.Tensor,
] = {
    "target_spacing_zyx_mm": torch.tensor(
        [3.0, 1.0, 1.0],
        dtype=torch.float32,
    ),  # [3]

    "patch_size_zyx": torch.tensor(
        [64, 128, 128],
        dtype=torch.int64,
    ),  # [3]

    "batch_size": torch.tensor(
        2,
        dtype=torch.int64,
    ),  # []
}


(
    planned_patch_fov_zyx_mm,
    planned_patch_voxel_count,
    planned_patch_volume_mm3,
) = compute_planned_patch_context(
    target_spacing_zyx_mm=experiment_plans[
        "target_spacing_zyx_mm"
    ],
    patch_size_zyx=experiment_plans[
        "patch_size_zyx"
    ],
)


# 관찰된 spacing과 결정된 spacing의 의미 차이 출력
print(
    "Observed median spacing:",
    dataset_fingerprint[
        "median_spacing_zyx_mm"
    ].tolist(),
    "mm",
)

print(
    "Planned target spacing:",
    experiment_plans[
        "target_spacing_zyx_mm"
    ].tolist(),
    "mm",
)


# Planned patch의 voxel-space·physical-space context 출력
print()
print(
    "Planned patch size:",
    experiment_plans[
        "patch_size_zyx"
    ].tolist(),
    "voxels",
)

print(
    "Planned patch FOV:",
    planned_patch_fov_zyx_mm.tolist(),
    "mm",
)

print(
    "Patch voxel count:",
    planned_patch_voxel_count,
)

print(
    "Physical patch volume:",
    planned_patch_volume_mm3.item(),
    "mm³",
)

print(
    "Planned batch size:",
    experiment_plans[
        "batch_size"
    ].item(),
)


# 같은 fingerprint에서 관찰값과 결정값을 구분하는 핵심 확인
print()
print(
    "Median spacing equals target spacing:",
    torch.equal(
        dataset_fingerprint[
            "median_spacing_zyx_mm"
        ],
        experiment_plans[
            "target_spacing_zyx_mm"
        ],
    ),
)

Observed median spacing: [3.25, 1.0, 1.0] mm
Planned target spacing: [3.0, 1.0, 1.0] mm

Planned patch size: [64, 128, 128] voxels
Planned patch FOV: [192.0, 128.0, 128.0] mm
Patch voxel count: 1048576
Physical patch volume: 3145728.0 mm³
Planned batch size: 2

Median spacing equals target spacing: False


In [11]:
# Cell 3 — Target Spacing과 Transpose

def transpose_spatial_geometry(
    original_shape: torch.Tensor,        # [3], integer
    original_spacing_mm: torch.Tensor,   # [3], floating-point
    transpose_forward: tuple[int, int, int],
) -> tuple[
    torch.Tensor,  # Transposed shape [3]
    torch.Tensor,  # Transposed spacing [3]
]:
    """동일한 axis permutation을 Shape와 spacing에 적용."""

    # 세 spatial axis를 가진 geometry인지 확인
    if original_shape.shape != (3,):
        raise ValueError(
            "original_shape의 Shape는 [3]이어야 합니다."
        )

    if original_spacing_mm.shape != (3,):
        raise ValueError(
            "original_spacing_mm의 Shape는 [3]이어야 합니다."
        )

    # 중복이나 누락 없는 axis permutation인지 확인
    if sorted(transpose_forward) != [0, 1, 2]:
        raise ValueError(
            "transpose_forward는 0, 1, 2의 permutation이어야 합니다."
        )

    # Shape와 spacing을 같은 순서로 재배치하기 위한 index 생성
    transpose_indices = torch.tensor(
        transpose_forward,
        dtype=torch.int64,
        device=original_shape.device,
    )  # [3]

    # Spatial array axis의 순서 변경
    transposed_shape = torch.index_select(
        original_shape,
        dim=0,
        index=transpose_indices,
    )  # [3]

    # 새 array axis와 spacing의 물리적 의미를 일치시키기 위한 재배치
    transposed_spacing_mm = torch.index_select(
        original_spacing_mm,
        dim=0,
        index=transpose_indices.to(
            original_spacing_mm.device
        ),
    )  # [3]

    return (
        transposed_shape,
        transposed_spacing_mm,
    )


def compute_resampled_shape(
    source_shape_zyx: torch.Tensor,       # [3], integer
    source_spacing_zyx_mm: torch.Tensor,  # [3], floating-point
    target_spacing_zyx_mm: torch.Tensor,  # [3], floating-point
) -> torch.Tensor:                       # [3], torch.int64
    """Physical extent를 보존하는 target voxel Shape 계산."""

    # 세 geometry Tensor의 동일한 spatial Shape 확인
    if (
        source_shape_zyx.shape != (3,)
        or source_spacing_zyx_mm.shape != (3,)
        or target_spacing_zyx_mm.shape != (3,)
    ):
        raise ValueError(
            "Shape와 spacing Tensor는 모두 [3]이어야 합니다."
        )

    # 0으로 나누거나 뒤집힌 geometry가 생기는 상황 차단
    if not bool(
        torch.all(source_shape_zyx > 0).item()
    ):
        raise ValueError(
            "모든 source Shape 값은 양수여야 합니다."
        )

    if not bool(
        torch.all(source_spacing_zyx_mm > 0).item()
        and torch.all(target_spacing_zyx_mm > 0).item()
    ):
        raise ValueError(
            "모든 spacing 값은 양수여야 합니다."
        )

    # 각 axis에서 source voxel 하나가 target voxel 몇 개에 해당하는지 계산
    resampling_scale_zyx = (
        source_spacing_zyx_mm
        / target_spacing_zyx_mm
    )  # [3]

    # 기존 physical extent를 target spacing의 voxel 개수로 변환
    resampled_shape_float = (
        source_shape_zyx.to(torch.float32)
        * resampling_scale_zyx
    )  # [3]

    # 실제 Tensor Shape로 사용할 수 있도록 반올림 후 integer 변환
    resampled_shape_zyx = torch.round(
        resampled_shape_float
    ).to(torch.int64)  # [3]

    return resampled_shape_zyx


# 원본 array의 axis별 voxel 개수와 spacing 구성
original_shape_xyz = torch.tensor(
    [160, 224, 96],
    dtype=torch.int64,
)  # [3]

original_spacing_xyz_mm = torch.tensor(
    [1.0, 1.0, 3.0],
    dtype=torch.float32,
)  # [3]


# 원본 (x, y, z)를 network 내부 (z, x, y) 순서로 변경
transpose_forward = (
    2,
    0,
    1,
)

(
    transposed_shape_zxy,
    transposed_spacing_zxy_mm,
) = transpose_spatial_geometry(
    original_shape=original_shape_xyz,
    original_spacing_mm=original_spacing_xyz_mm,
    transpose_forward=transpose_forward,
)


# Network 내부 axis 순서에 대응하는 target spacing 구성
target_spacing_zxy_mm = torch.tensor(
    [2.5, 1.0, 1.0],
    dtype=torch.float32,
)  # [3]


# Target spacing에서 필요한 새로운 voxel Shape 계산
resampled_shape_zxy = compute_resampled_shape(
    source_shape_zyx=transposed_shape_zxy,
    source_spacing_zyx_mm=transposed_spacing_zxy_mm,
    target_spacing_zyx_mm=target_spacing_zxy_mm,
)  # [3]


# Transpose 전후의 axis 이름 추적
original_axis_names = (
    "x",
    "y",
    "z",
)

transposed_axis_names = tuple(
    original_axis_names[axis_index]
    for axis_index in transpose_forward
)


# Resampling 전후의 physical extent 계산
source_extent_zxy_mm = (
    transposed_shape_zxy.to(torch.float32)
    * transposed_spacing_zxy_mm
)  # [3]

resampled_extent_zxy_mm = (
    resampled_shape_zxy.to(torch.float32)
    * target_spacing_zxy_mm
)  # [3]


print(
    "Original axes:",
    original_axis_names,
)

print(
    "Transposed axes:",
    transposed_axis_names,
)

print()
print(
    "Original shape:",
    original_shape_xyz.tolist(),
)

print(
    "Transposed shape:",
    transposed_shape_zxy.tolist(),
)

print(
    "Transposed spacing:",
    transposed_spacing_zxy_mm.tolist(),
    "mm",
)

print()
print(
    "Target spacing:",
    target_spacing_zxy_mm.tolist(),
    "mm",
)

print(
    "Resampled shape:",
    resampled_shape_zxy.tolist(),
)

print(
    "Source physical extent:",
    source_extent_zxy_mm.tolist(),
    "mm",
)

print(
    "Resampled physical extent:",
    resampled_extent_zxy_mm.tolist(),
    "mm",
)

Original axes: ('x', 'y', 'z')
Transposed axes: ('z', 'x', 'y')

Original shape: [160, 224, 96]
Transposed shape: [96, 160, 224]
Transposed spacing: [3.0, 1.0, 1.0] mm

Target spacing: [2.5, 1.0, 1.0] mm
Resampled shape: [115, 160, 224]
Source physical extent: [288.0, 160.0, 224.0] mm
Resampled physical extent: [287.5, 160.0, 224.0] mm


In [12]:
# Cell 4 — Patch·Stride·Architecture 제약

def compute_stride_product(
    stage_strides_zyx: torch.Tensor,  # [S, 3], integer
) -> torch.Tensor:                   # [3], integer
    """모든 downsampling stage의 axis별 누적 stride 계산."""

    # Stage와 spatial axis를 가진 2D Tensor인지 확인
    if (
        stage_strides_zyx.ndim != 2
        or stage_strides_zyx.shape[1] != 3
    ):
        raise ValueError(
            "stage_strides_zyx의 Shape는 [S, 3]이어야 합니다."
        )

    # Spatial downsampling 배수를 나타내는 integer인지 확인
    if torch.is_floating_point(
        stage_strides_zyx
    ):
        raise TypeError(
            "stage stride는 integer Tensor여야 합니다."
        )

    # 0 이하의 잘못된 stride 차단
    if not bool(
        torch.all(stage_strides_zyx > 0).item()
    ):
        raise ValueError(
            "모든 stage stride는 양수여야 합니다."
        )

    # 모든 stage의 stride를 axis별로 곱하여 전체 축소 배수 계산
    stride_product_zyx = torch.prod(
        stage_strides_zyx,
        dim=0,
    )  # [3]

    return stride_product_zyx


def check_patch_stride_compatibility(
    patch_size_zyx: torch.Tensor,      # [3], integer
    stride_product_zyx: torch.Tensor,  # [3], integer
) -> tuple[
    bool,          # 모든 axis의 호환 여부
    torch.Tensor,  # Axis별 나머지 [3]
]:
    """Patch size와 누적 stride의 axis별 divisibility 확인."""

    # Patch와 stride product의 spatial Shape 확인
    if (
        patch_size_zyx.shape != (3,)
        or stride_product_zyx.shape != (3,)
    ):
        raise ValueError(
            "patch size와 stride product의 Shape는 [3]이어야 합니다."
        )

    # 각 axis에서 나누어떨어지는지 확인하기 위한 나머지 계산
    axis_remainders_zyx = torch.remainder(
        patch_size_zyx,
        stride_product_zyx,
    )  # [3]

    # 세 axis의 나머지가 모두 0인지 확인
    is_compatible = bool(
        torch.all(
            axis_remainders_zyx == 0
        ).item()
    )

    return (
        is_compatible,
        axis_remainders_zyx,
    )


def trace_encoder_spatial_shapes(
    patch_size_zyx: torch.Tensor,   # [3], integer
    stage_strides_zyx: torch.Tensor,  # [S, 3], integer
) -> torch.Tensor:                 # [S+1, 3], integer
    """입력 patch부터 bottleneck까지의 spatial Shape 추적."""

    # 입력 Shape를 첫 번째 trace 항목으로 저장
    spatial_shape_trace: list[
        torch.Tensor
    ] = [
        patch_size_zyx.clone()
    ]

    current_shape_zyx = (
        patch_size_zyx.clone()
    )  # [3]

    # 각 downsampling stage를 순서대로 적용
    for stage_stride_zyx in stage_strides_zyx:
        # 현재 Shape가 해당 stage stride로 나누어지는지 확인
        stage_remainder_zyx = torch.remainder(
            current_shape_zyx,
            stage_stride_zyx,
        )  # [3]

        if not bool(
            torch.all(
                stage_remainder_zyx == 0
            ).item()
        ):
            raise ValueError(
                "현재 spatial Shape가 stage stride와 호환되지 않습니다."
            )

        # Strided operation 이후의 spatial Shape 계산
        current_shape_zyx = torch.div(
            current_shape_zyx,
            stage_stride_zyx,
            rounding_mode="floor",
        )  # [3]

        spatial_shape_trace.append(
            current_shape_zyx.clone()
        )

    # [3] Tensor 목록을 [S+1, 3] Tensor로 결합
    return torch.stack(
        spatial_shape_trace,
        dim=0,
    )


# Anisotropic z축을 초기에 보존하는 교육용 stride 구성
stage_strides_zyx = torch.tensor(
    [
        [1, 2, 2],
        [2, 2, 2],
        [2, 2, 2],
    ],
    dtype=torch.int64,
)  # [S=3, 3]


# 모든 encoder stage를 통과한 누적 downsampling 배수 계산
stride_product_zyx = compute_stride_product(
    stage_strides_zyx=stage_strides_zyx,
)  # [3]


# 누적 stride와 호환되는 patch와 호환되지 않는 patch 구성
compatible_patch_size_zyx = torch.tensor(
    [64, 128, 128],
    dtype=torch.int64,
)  # [3]

incompatible_patch_size_zyx = torch.tensor(
    [62, 128, 128],
    dtype=torch.int64,
)  # [3]


# 두 patch의 axis별 나머지와 전체 호환 여부 계산
(
    compatible_result,
    compatible_remainders,
) = check_patch_stride_compatibility(
    patch_size_zyx=compatible_patch_size_zyx,
    stride_product_zyx=stride_product_zyx,
)

(
    incompatible_result,
    incompatible_remainders,
) = check_patch_stride_compatibility(
    patch_size_zyx=incompatible_patch_size_zyx,
    stride_product_zyx=stride_product_zyx,
)


# 호환되는 patch의 encoder spatial Shape 변화 추적
encoder_shape_trace = trace_encoder_spatial_shapes(
    patch_size_zyx=compatible_patch_size_zyx,
    stage_strides_zyx=stage_strides_zyx,
)  # [S+1=4, 3]


print(
    "Stage strides:",
    stage_strides_zyx.tolist(),
)

print(
    "Stride product:",
    stride_product_zyx.tolist(),
)

print()
print(
    "Compatible patch:",
    compatible_patch_size_zyx.tolist(),
    "| remainder:",
    compatible_remainders.tolist(),
    "| compatible:",
    compatible_result,
)

print(
    "Incompatible patch:",
    incompatible_patch_size_zyx.tolist(),
    "| remainder:",
    incompatible_remainders.tolist(),
    "| compatible:",
    incompatible_result,
)


# 입력부터 bottleneck까지 stage별 spatial Shape 출력
print()
print("Encoder spatial Shape trace")

for stage_index, spatial_shape_zyx in enumerate(
    encoder_shape_trace
):
    print(
        f"Stage {stage_index}:",
        spatial_shape_zyx.tolist(),
    )

Stage strides: [[1, 2, 2], [2, 2, 2], [2, 2, 2]]
Stride product: [4, 8, 8]

Compatible patch: [64, 128, 128] | remainder: [0, 0, 0] | compatible: True
Incompatible patch: [62, 128, 128] | remainder: [2, 0, 0] | compatible: False

Encoder spatial Shape trace
Stage 0: [64, 128, 128]
Stage 1: [64, 64, 64]
Stage 2: [32, 32, 32]
Stage 3: [16, 16, 16]


In [13]:
# Cell 5 — Preprocessing과 Frozen Experiment Boundary

def experiment_values_match(
    reference_value: object,
    candidate_value: object,
) -> bool:
    """Tensor와 일반 Python 값의 동일성 비교."""

    # 두 값이 모두 Tensor일 때 Shape·dtype·값의 정확한 일치 확인
    if (
        isinstance(reference_value, torch.Tensor)
        and isinstance(candidate_value, torch.Tensor)
    ):
        return bool(
            torch.equal(
                reference_value,
                candidate_value,
            )
        )

    # 한쪽만 Tensor인 서로 다른 자료형 차단
    if (
        isinstance(reference_value, torch.Tensor)
        or isinstance(candidate_value, torch.Tensor)
    ):
        return False

    # 문자열·정수·tuple 등 일반 Python 값 비교
    return bool(
        reference_value == candidate_value
    )


def audit_experiment_boundary(
    reference_settings: dict[str, object],
    candidate_settings: dict[str, object],
    frozen_field_names: tuple[str, ...],
    allowed_changed_field_names: tuple[str, ...],
) -> tuple[
    bool,            # 공정한 비교 여부
    tuple[str, ...], # 실제로 변경된 field
    tuple[str, ...], # Frozen boundary 위반 field
    tuple[str, ...], # 허용 목록 밖의 변경 field
]:
    """Reference와 candidate의 실험 통제 경계 감사."""

    # 두 설정에 등장하는 모든 field 이름 수집
    all_field_names = sorted(
        set(reference_settings)
        | set(candidate_settings)
    )

    changed_field_names: list[str] = []

    # 누락되거나 서로 다른 field 탐색
    for field_name in all_field_names:
        if (
            field_name not in reference_settings
            or field_name not in candidate_settings
        ):
            changed_field_names.append(
                field_name
            )

            continue

        values_are_equal = experiment_values_match(
            reference_value=reference_settings[
                field_name
            ],
            candidate_value=candidate_settings[
                field_name
            ],
        )

        if not values_are_equal:
            changed_field_names.append(
                field_name
            )

    # 고정해야 하지만 실제로 변경된 confounding field 탐색
    frozen_violations = tuple(
        field_name
        for field_name in changed_field_names
        if field_name in frozen_field_names
    )

    # 변경 허용 목록에도 고정 목록에도 없는 예상 밖의 변경 탐색
    unexpected_changes = tuple(
        field_name
        for field_name in changed_field_names
        if (
            field_name
            not in allowed_changed_field_names
            and field_name
            not in frozen_field_names
        )
    )

    # Frozen 위반이 없고 sampling policy만 변경된 경우 통과
    comparison_is_fair = (
        len(frozen_violations) == 0
        and len(unexpected_changes) == 0
        and set(changed_field_names).issubset(
            allowed_changed_field_names
        )
    )

    return (
        comparison_is_fair,
        tuple(changed_field_names),
        frozen_violations,
        unexpected_changes,
    )


# nnU-Net preprocessing의 단순화된 실행 순서 기록
preprocessing_stages: tuple[str, ...] = (
    "load_image_and_label",
    "transpose_spatial_axes",
    "crop_to_nonzero_region",
    "resample_image",
    "resample_label",
    "normalize_image",
    "save_preprocessed_case",
)


# 모든 sampling method에서 고정할 실험 조건 정의
frozen_field_names: tuple[str, ...] = (
    "data_split",
    "target_spacing_zyx_mm",
    "transpose_forward",
    "patch_size_zyx",
    "batch_size",
    "architecture_strides_zyx",
    "normalization_scheme",
    "image_resampling",
    "label_resampling",
    "training_update_budget",
)


# OLES3D 연구에서 의도적으로 변경할 유일한 조건 정의
allowed_changed_field_names: tuple[str, ...] = (
    "sampling_policy",
)


# Unmodified nnU-Net reference의 실험 설정 구성
reference_settings: dict[str, object] = {
    "data_split": "fold_0",
    "target_spacing_zyx_mm": torch.tensor(
        [3.0, 1.0, 1.0],
        dtype=torch.float32,
    ),  # [3]

    "transpose_forward": (
        2,
        0,
        1,
    ),

    "patch_size_zyx": torch.tensor(
        [64, 128, 128],
        dtype=torch.int64,
    ),  # [3]

    "batch_size": 2,

    "architecture_strides_zyx": torch.tensor(
        [
            [1, 2, 2],
            [2, 2, 2],
            [2, 2, 2],
        ],
        dtype=torch.int64,
    ),  # [S=3, 3]

    "normalization_scheme": "CTNormalization",
    "image_resampling": "continuous",
    "label_resampling": "discrete",
    "training_update_budget": 10000,
    "sampling_policy": "nnunet_default",
}


# 동일한 조건에서 sampling policy만 변경한 OLES3D 설정 구성
oles3d_settings: dict[str, object] = {
    **reference_settings,
    "sampling_policy": "oles3d_adaptive",
}


# Sampling policy와 patch size를 동시에 변경한 confounded 설정 구성
confounded_settings: dict[str, object] = {
    **oles3d_settings,
    "patch_size_zyx": torch.tensor(
        [48, 96, 96],
        dtype=torch.int64,
    ),  # [3]
}


# 공정한 OLES3D 비교 설정 감사
(
    oles3d_is_fair,
    oles3d_changed_fields,
    oles3d_frozen_violations,
    oles3d_unexpected_changes,
) = audit_experiment_boundary(
    reference_settings=reference_settings,
    candidate_settings=oles3d_settings,
    frozen_field_names=frozen_field_names,
    allowed_changed_field_names=(
        allowed_changed_field_names
    ),
)


# Patch size까지 변경된 confounded 설정 감사
(
    confounded_is_fair,
    confounded_changed_fields,
    confounded_frozen_violations,
    confounded_unexpected_changes,
) = audit_experiment_boundary(
    reference_settings=reference_settings,
    candidate_settings=confounded_settings,
    frozen_field_names=frozen_field_names,
    allowed_changed_field_names=(
        allowed_changed_field_names
    ),
)


# Preprocessing stage 순서 출력
print("Preprocessing data flow")

for (
    stage_index,
    preprocessing_stage,
) in enumerate(
    preprocessing_stages,
    start=1,
):
    print(
        f"{stage_index}. {preprocessing_stage}"
    )


# Sampling policy만 변경한 공정한 비교 결과 출력
print()
print("OLES3D comparison")
print(
    "Changed fields:",
    oles3d_changed_fields,
)
print(
    "Frozen violations:",
    oles3d_frozen_violations,
)
print(
    "Unexpected changes:",
    oles3d_unexpected_changes,
)
print(
    "Fair comparison:",
    oles3d_is_fair,
)


# Patch size가 함께 변경된 confounded 비교 결과 출력
print()
print("Confounded comparison")
print(
    "Changed fields:",
    confounded_changed_fields,
)
print(
    "Frozen violations:",
    confounded_frozen_violations,
)
print(
    "Unexpected changes:",
    confounded_unexpected_changes,
)
print(
    "Fair comparison:",
    confounded_is_fair,
)

Preprocessing data flow
1. load_image_and_label
2. transpose_spatial_axes
3. crop_to_nonzero_region
4. resample_image
5. resample_label
6. normalize_image
7. save_preprocessed_case

OLES3D comparison
Changed fields: ('sampling_policy',)
Frozen violations: ()
Unexpected changes: ()
Fair comparison: True

Confounded comparison
Changed fields: ('patch_size_zyx', 'sampling_policy')
Frozen violations: ('patch_size_zyx',)
Unexpected changes: ()
Fair comparison: False
